In [3]:
import pandas as pd
import numpy as np
from functools import reduce
from itertools import groupby
from operator import itemgetter
from pathlib import Path

# Load data
data_path = Path('..') / 'data' / 'cleaned' / 'amazon_products_sales_data_cleaned.csv'
df = pd.read_csv(data_path)
df['data_collected_at'] = pd.to_datetime(df['data_collected_at'])

print(f"Dataset: {len(df)} products")
print(f"Categories: {', '.join(df['product_category'].unique()[:5])}...")

Dataset: 42675 products
Categories: Phones, Laptops, Other Electronics, Cameras, Storage...


In [4]:
# Pricing strategy analysis using stream operations and lambda expressions
# Filter -> Map -> GroupBy -> Aggregate pipeline

pricing_tiers = (
    df[df['discounted_price'].notna()] # type: ignore
    .assign(
        price_tier=lambda x: pd.cut( # pyright: ignore[reportUndefinedVariable]
            x['discounted_price'],
            bins=[0, 25, 100, 500, float('inf')],
            labels=['Budget', 'Mid-Range', 'Premium', 'Luxury']
        ),
        revenue_potential=lambda x: x['discounted_price'] * x['purchased_last_month'].fillna(0)
    )
    .groupby(['product_category', 'price_tier'])
    .agg({
        'product_title': 'count',
        'product_rating': 'mean',
        'revenue_potential': 'sum',
        'discount_percentage': 'mean'
    })
    .rename(columns={'product_title': 'product_count'})
    .round(2)
)

print("\nPricing Strategy by Category and Tier:")
print(pricing_tiers.sort_values('revenue_potential', ascending=False).head(15))


Pricing Strategy by Category and Tier:
                              product_count  product_rating  \
product_category  price_tier                                  
Power & Batteries Budget               1439            4.76   
                  Mid-Range             992            4.52   
Phones            Premium              2103            4.26   
Laptops           Premium              2941            4.28   
                  Luxury               2205            4.26   
                  Mid-Range            1969            4.51   
Other Electronics Premium              2437            4.51   
Phones            Mid-Range            1912            4.23   
Other Electronics Mid-Range            3097            4.39   
Cameras           Premium              1122            4.54   
Other Electronics Budget               2619            4.42   
TV & Display      Premium               655            4.67   
Cameras           Mid-Range            1330            4.56   
Phones         

C:\Users\HP\AppData\Local\Temp\ipykernel_26612\3173455514.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(['product_category', 'price_tier'])


In [5]:
# Multi-dimensional aggregation: Review engagement metrics
# Using functional approach with map, filter, and custom aggregators

# Define custom aggregation functions
review_metrics = (
    df[df['total_reviews'].notna() & (df['total_reviews'] > 0)] # type: ignore
    .assign(
        engagement_score=lambda x: np.log1p(x['total_reviews']) * x['product_rating'], # pyright: ignore[reportUndefinedVariable]
        review_intensity=lambda x: x['total_reviews'] / x['purchased_last_month'].fillna(1).clip(lower=1)
    )
    .groupby('product_category')
    .agg({
        'engagement_score': ['mean', 'max', 'std'],
        'review_intensity': ['mean', 'median'],
        'total_reviews': ['sum', lambda x: x.quantile(0.75)],
        'product_rating': lambda x: (x >= 4.5).sum()  # Count high-rated products
    })
)

# Flatten and clean column names
review_metrics.columns = ['_'.join(map(str, col)).strip('_') for col in review_metrics.columns]
review_metrics = review_metrics.rename(columns={
    'total_reviews_<lambda_0>': 'reviews_75th_percentile',
    'product_rating_<lambda_0>': 'high_rated_count'
})

print("\nReview Engagement Analysis:")
print(review_metrics.round(2))


Review Engagement Analysis:
                     engagement_score_mean  engagement_score_max  \
product_category                                                   
Cameras                              24.74                 55.53   
Chargers & Cables                    24.81                 62.87   
Gaming                               31.76                 54.46   
Headphones                           29.31                 49.74   
Laptops                              23.30                 64.25   
Networking                           24.23                 55.46   
Other Electronics                    26.39                 58.06   
Phones                               28.49                 54.03   
Power & Batteries                    34.20                 56.35   
Printers & Scanners                  26.55                 58.55   
Smart Home                           27.45                 61.84   
Speakers                             25.02                 54.29   
Storage            

In [6]:
# Sponsored vs Organic performance using reduce and functional composition
# Demonstrates stream-like data transformation pipeline

def calculate_performance_metrics(group):
    """Functional aggregator for performance metrics"""
    return pd.Series({ # type: ignore
        'avg_rating': group['product_rating'].mean(),
        'median_price': group['discounted_price'].median(),
        'total_purchases': group['purchased_last_month'].sum(),
        'avg_reviews': group['total_reviews'].mean(),
        'product_count': len(group)
    })

# Stream-like pipeline: filter, group, map, reduce
sponsorship_analysis = (
    df[df['is_sponsored'].isin(['Sponsored', 'Organic'])] # type: ignore
    .groupby(['product_category', 'is_sponsored'])
    .apply(calculate_performance_metrics, include_groups=False)
    .reset_index()
    .pivot(index='product_category', columns='is_sponsored', values=['avg_rating', 'median_price', 'total_purchases'])
)

# Calculate performance differential using lambda
if ('avg_rating', 'Sponsored') in sponsorship_analysis.columns and ('avg_rating', 'Organic') in sponsorship_analysis.columns:
    sponsorship_analysis[('rating_diff', '')] = sponsorship_analysis.apply(
        lambda row: row[('avg_rating', 'Sponsored')] - row[('avg_rating', 'Organic')]
        if pd.notna(row[('avg_rating', 'Sponsored')]) and pd.notna(row[('avg_rating', 'Organic')]) else np.nan, # type: ignore
        axis=1
    )

print("\nSponsored vs Organic Performance:")
print(sponsorship_analysis.round(2))


Sponsored vs Organic Performance:
                    avg_rating           median_price            \
is_sponsored           Organic Sponsored      Organic Sponsored   
product_category                                                  
Cameras                   4.53      4.31       115.13    299.00   
Chargers & Cables         4.44      4.22        32.99     79.99   
Gaming                    4.27      4.74       149.99    309.99   
Headphones                4.34      3.69       139.95    124.99   
Laptops                   4.33      4.39       248.99     59.99   
Networking                4.22      4.27       109.00    159.99   
Other Electronics         4.44      4.31        59.24    400.99   
Phones                    4.28      4.39        57.99     79.95   
Power & Batteries         4.46      4.73        28.91     19.69   
Printers & Scanners       4.33       NaN        80.05       NaN   
Smart Home                4.47      4.86       252.02    399.99   
Speakers                  4

In [7]:
# Advanced reduce operation: Cross-category competitive analysis
# Using reduce to aggregate competitive metrics

# Prepare data for reduce operation
category_products = df.groupby('product_category').apply( # type: ignore
    lambda g: g.nlargest(5, 'total_reviews')[['product_rating', 'discounted_price', 'total_reviews']].to_dict('records'),
    include_groups=False
).to_dict()

# Define reduce function for competitive analysis
def competitive_reducer(acc, item):
    category, products = item
    if products:
        ratings = [p['product_rating'] for p in products if pd.notna(p.get('product_rating'))] # type: ignore
        prices = [p['discounted_price'] for p in products if pd.notna(p.get('discounted_price'))] # type: ignore
        reviews = [p['total_reviews'] for p in products if pd.notna(p.get('total_reviews'))] # type: ignore
        
        if ratings and prices and reviews:
            acc[category] = {
                'avg_top_rating': np.mean(ratings), # pyright: ignore[reportUndefinedVariable]
                'avg_top_price': np.mean(prices),
                'avg_top_reviews': np.mean(reviews),
                'competitive_index': np.mean(ratings) * np.log1p(np.mean(reviews)) / max(np.mean(prices), 1)
            }
    return acc

# Apply reduce operation
competitive_metrics = reduce(competitive_reducer, category_products.items(), {})
competitive_df = pd.DataFrame(competitive_metrics).T.round(2)

print("\nCompetitive Analysis (Top 5 Products per Category):")
print(competitive_df.sort_values('competitive_index', ascending=False))


Competitive Analysis (Top 5 Products per Category):
                     avg_top_rating  avg_top_price  avg_top_reviews  \
Power & Batteries              4.64          13.78         155879.2   
Phones                         4.40          20.06         139943.0   
Laptops                        4.56          23.07         401652.4   
Chargers & Cables              4.72          27.59         417639.4   
TV & Display                   4.70          30.22         114882.0   
Other Electronics              4.76          31.92         166218.6   
Networking                     3.98          27.15         122010.0   
Storage                        4.64          34.30         320286.0   
Headphones                     4.40          28.75          54357.6   
Cameras                        4.60          32.68         130139.8   
Smart Home                     4.54          46.03         121257.0   
Wearables                      4.60          46.51          31423.2   
Printers & Scanners     